In [1]:
%load_ext autoreload
%autoreload 2

import sys, os, json
dir = os.getcwd()

ext = ['', '/..', '/../..', '/../../src/models', '/../../src/nlp', '/../../src/synth']
sys.path += [dir + i for i in ext]

In [2]:
from eval import *
from state import *
from expectation import *

In [3]:
from api.football import *

In [4]:
# Get folder path
folder_path = os.path.join('data')

demos = []
for i in os.listdir(folder_path):
    if i.startswith('demonstration0'): demos += [i]
    
demos.sort()

print(demos)

['demonstration0']


In [5]:
import json
from scene import Scene
from api.objects.registry import REGISTRY

scenes = {}

i = 0
for d in demos:
    if not i: # i=0 for first demo
        print('initial demo (w/ language)')
    else:
        print(f'helper demo {i}')

    folder = os.path.join(folder_path, d, 'json_segments')
    dir_list = [i for i in os.listdir(folder) if i.endswith('.json')]
    dir_list.sort()

    # print(dir_list)

    j = 0
    for f in dir_list:

        file = os.path.join(folder, f)

        # print(file)

        with open(file) as f:
            data = json.load(f)
        
        # for t in data['scene']:
        #     print(t)
        
        if not i:
            scenes[j] = Scene.from_dict(data['scene'], REGISTRY)
        else:
            try:
                scenes[j].add_demo(Scene.from_dict(data['scene'], REGISTRY))
            except KeyError:
                print(f"Attempted to access part {j} of a demonstration. Either non-existant or not in the original demo.")

        print(f'part {j} -> {scenes[j]}')

        j += 1
    

    i += 1
    print()

initial demo (w/ language)
part 0 -> <scene.Scene object at 0x13247f750>



In [6]:
objects = {obj.id: obj for obj in scenes[0].allObjects}

for id in objects.keys():
    print(id)

corner1
corner2
corner3
corner4
coach
midfielder2
rightBack
midfielder1
centerBack
opponent_A
opponent_B
opponent_C
opponent_D
opponent_E
goalkeeper
ball
goal
goal_leftpost
goal_rightpost
target


In [7]:
mid_moved_down = MovedToBox(objects['midfielder2'], (-10, 10), (-16, -4))
coach_has_ball = HasBallPosession(objects['coach'])
mid2_has_ball = HasBallPosession(objects['midfielder2'])
mid1_has_ball = HasBallPosession(objects['midfielder1'])

In [8]:
coach_received_ball = DidHappen([
    coach_has_ball,
    (coach_has_ball, False)
])

mid2_received_ball = DidHappen([
    coach_has_ball,
    (coach_has_ball, False),
    mid2_has_ball
])

mid1_received_ball = DidHappen([
    mid2_has_ball,
    (mid2_has_ball, False),
    mid1_has_ball
])

mid_ever_down = DidHappen(mid_moved_down)

In [9]:
from eval import Eval
eval = Eval(scenes)

In [10]:
eval.sub([
    mid_moved_down,
    coach_has_ball,
    mid2_has_ball,
    mid1_has_ball
])

eval.verify([
    coach_received_ball,
    mid2_received_ball,
    mid1_received_ball
])

In [11]:
score = eval.run()

[1, 1, 0]
Score: 2/3


In [12]:
eval.timeline

[(coach has ball posession, True),
 (coach has ball posession, False),
 (midfielder2 has ball posession, True),
 (midfielder2 has ball posession, False),
 (midfielder2 moved to bounds within ((-10, 10), (-16, -4)), True)]